# Homework 4: Feature engineering/selection & stacking


## Overview

This notebook is organized around the assignment requirements.

Goals:
1. build multiple tuned models that differ in a meaningful way
2. create and test new features
3. evaluate which features are useful
4. combine models with an ensemble
5. discuss what worked, what did not work, and what to continue using


## 1. Imports and setup

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import balanced_accuracy_score, log_loss, classification_report
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.base import clone


## 2. Load the competition files

In [2]:
train_candidates = [
    "/Users/christinewu/Downloads/playground-series-s6e4/train.csv",
    "train.csv",
    "./train.csv"
]

test_candidates = [
    "/Users/christinewu/Downloads/playground-series-s6e4/test.csv",
    "test.csv",
    "./test.csv"
]

sample_candidates = [
    "/Users/christinewu/Downloads/playground-series-s6e4/sample_submission.csv",
    "sample_submission.csv",
    "./sample_submission.csv"
]

def first_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = first_existing_path(train_candidates)
test_path = first_existing_path(test_candidates)
sample_path = first_existing_path(sample_candidates)

if train_path is None or test_path is None or sample_path is None:
    raise FileNotFoundError(
        "Could not find train.csv, test.csv, and sample_submission.csv at the expected path."
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train path:", train_path)
print("Test path:", test_path)
print("Sample path:", sample_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

train.head()


Train path: /Users/christinewu/Downloads/playground-series-s6e4/train.csv
Test path: /Users/christinewu/Downloads/playground-series-s6e4/test.csv
Sample path: /Users/christinewu/Downloads/playground-series-s6e4/sample_submission.csv
Train shape: (630000, 21)
Test shape: (270000, 20)
Sample submission shape: (270000, 2)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


## 3. Inspect the data

In [3]:
print("Train columns:")
print(train.columns.tolist())
print()

print("Target distribution:")
display(train["Irrigation_Need"].value_counts())

print("Training data types:")
display(train.dtypes)


Train columns:
['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']

Target distribution:


Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Training data types:


id                           int64
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object


## 4. Prepare the target and feature matrix

The target is encoded with LabelEncoder so the models can work with integer labels.
The original class names are saved so predictions can be converted back for submission.


In [4]:
target_col = "Irrigation_Need"
id_col = "id"

le = LabelEncoder()
y = le.fit_transform(train[target_col])

X = train.drop(columns=[target_col]).copy()
X_test = test.copy()

class_names = list(le.classes_)
n_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", n_classes)


Classes: ['High', 'Low', 'Medium']
Number of classes: 3



## 5. Separate numeric and categorical variables

This dataset has a mix of numeric, categorical, and binary indicator style columns.
I will use this distinction for preprocessing and feature engineering.


In [5]:
feature_cols = [c for c in X.columns if c != id_col]
X = X[feature_cols]
X_test = X_test[feature_cols]

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print()
print("Categorical columns:", categorical_cols)


Numeric columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']



## 6. Baseline preprocessing

Baseline preprocessing uses:
- median imputation for numeric variables
- most frequent imputation for categorical variables
- one-hot encoding for categorical variables

This gives a simple representation for the first round of models.


In [7]:
base_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols)
    ],
    remainder="drop"
)



## 7. Create and test new features

This notebook tests several feature engineering ideas:
- agronomic ratios
- moisture and climate interaction terms
- rainfall and irrigation balance features
- per-area style scaling features
- log transforms for selected positive features
- categorical combination features
- frequency encoding for categorical variables

These features are intended to test whether domain-informed structure improves multiclass prediction.


In [8]:
X_fe = X.copy()
X_test_fe = X_test.copy()

# Numeric interactions and ratios based on the real columns
X_fe["moisture_x_temp"] = X_fe["Soil_Moisture"] * X_fe["Temperature_C"]
X_test_fe["moisture_x_temp"] = X_test_fe["Soil_Moisture"] * X_test_fe["Temperature_C"]

X_fe["humidity_x_temp"] = X_fe["Humidity"] * X_fe["Temperature_C"]
X_test_fe["humidity_x_temp"] = X_test_fe["Humidity"] * X_test_fe["Temperature_C"]

X_fe["rainfall_plus_prev_irrigation"] = X_fe["Rainfall_mm"] + X_fe["Previous_Irrigation_mm"]
X_test_fe["rainfall_plus_prev_irrigation"] = X_test_fe["Rainfall_mm"] + X_test_fe["Previous_Irrigation_mm"]

X_fe["rainfall_minus_prev_irrigation"] = X_fe["Rainfall_mm"] - X_fe["Previous_Irrigation_mm"]
X_test_fe["rainfall_minus_prev_irrigation"] = X_test_fe["Rainfall_mm"] - X_test_fe["Previous_Irrigation_mm"]

X_fe["water_input_per_area"] = (
    X_fe["Rainfall_mm"] + X_fe["Previous_Irrigation_mm"]
) / (X_fe["Field_Area_hectare"] + 1e-6)
X_test_fe["water_input_per_area"] = (
    X_test_fe["Rainfall_mm"] + X_test_fe["Previous_Irrigation_mm"]
) / (X_test_fe["Field_Area_hectare"] + 1e-6)

X_fe["sunlight_per_temp"] = X_fe["Sunlight_Hours"] / (X_fe["Temperature_C"].abs() + 1e-6)
X_test_fe["sunlight_per_temp"] = X_test_fe["Sunlight_Hours"] / (X_test_fe["Temperature_C"].abs() + 1e-6)

X_fe["ph_x_ec"] = X_fe["Soil_pH"] * X_fe["Electrical_Conductivity"]
X_test_fe["ph_x_ec"] = X_test_fe["Soil_pH"] * X_test_fe["Electrical_Conductivity"]

X_fe["organic_x_moisture"] = X_fe["Organic_Carbon"] * X_fe["Soil_Moisture"]
X_test_fe["organic_x_moisture"] = X_test_fe["Organic_Carbon"] * X_test_fe["Soil_Moisture"]

X_fe["wind_x_sunlight"] = X_fe["Wind_Speed_kmh"] * X_fe["Sunlight_Hours"]
X_test_fe["wind_x_sunlight"] = X_test_fe["Wind_Speed_kmh"] * X_test_fe["Sunlight_Hours"]

# Log transforms
for col in ["Rainfall_mm", "Previous_Irrigation_mm", "Field_Area_hectare", "Electrical_Conductivity"]:
    X_fe[f"{col}_log1p"] = np.log1p(X_fe[col].clip(lower=0))
    X_test_fe[f"{col}_log1p"] = np.log1p(X_test_fe[col].clip(lower=0))

# Row-level summary features across core weather and soil variables
summary_cols = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]

X_fe["core_mean"] = X_fe[summary_cols].mean(axis=1)
X_test_fe["core_mean"] = X_test_fe[summary_cols].mean(axis=1)

X_fe["core_std"] = X_fe[summary_cols].std(axis=1)
X_test_fe["core_std"] = X_test_fe[summary_cols].std(axis=1)

# Frequency encoding
for col in categorical_cols:
    freq_map = X_fe[col].value_counts(normalize=True).to_dict()
    X_fe[f"{col}_freq"] = X_fe[col].map(freq_map)
    X_test_fe[f"{col}_freq"] = X_test_fe[col].map(freq_map).fillna(0)

# Categorical combinations
pair_cols = [
    ("Soil_Type", "Crop_Type"),
    ("Season", "Region"),
    ("Irrigation_Type", "Water_Source"),
    ("Crop_Growth_Stage", "Season"),
    ("Mulching_Used", "Crop_Type"),
]

for c1, c2 in pair_cols:
    new_col = f"{c1}__{c2}"
    X_fe[new_col] = X_fe[c1].astype(str) + "_" + X_fe[c2].astype(str)
    X_test_fe[new_col] = X_test_fe[c1].astype(str) + "_" + X_test_fe[c2].astype(str)

print("Original feature count:", X.shape[1])
print("Engineered feature count:", X_fe.shape[1])


Original feature count: 19
Engineered feature count: 47



## 8. Re-detect feature types after feature engineering

Feature engineering adds both numeric and categorical features, so the preprocessing object must be rebuilt.


In [9]:
numeric_cols_fe = X_fe.select_dtypes(include=["number"]).columns.tolist()
categorical_cols_fe = X_fe.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric columns after feature engineering:", len(numeric_cols_fe))
print("Categorical columns after feature engineering:", len(categorical_cols_fe))


Numeric columns after feature engineering: 34
Categorical columns after feature engineering: 13



## 9. Build preprocessing objects for the baseline and engineered feature sets

This allows direct comparison between original and engineered representations.


In [10]:
base_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols)
    ],
    remainder="drop"
)

fe_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_cols_fe),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols_fe)
    ],
    remainder="drop"
)



## 10. Build multiple models that differ in a meaningful way

The models differ by model family and feature representation:
- ExtraTrees on the original feature set
- HistGradientBoosting on the engineered feature set
- LogisticRegression on the engineered feature set with scaling

This provides variety for later ensembling without using overly heavy settings.


In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_et = Pipeline(steps=[
    ("preprocessor", base_preprocessor),
    ("model", ExtraTreesClassifier(
        n_estimators=220,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

model_hgb = Pipeline(steps=[
    ("preprocessor", fe_preprocessor),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_depth=8,
        max_iter=280,
        min_samples_leaf=25,
        l2_regularization=0.0,
        random_state=42
    ))
])

model_lr = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols_fe),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]), categorical_cols_fe)
        ],
        remainder="drop"
    )),
    ("model", LogisticRegression(
        C=1.0,
        max_iter=2000,
        multi_class="auto"
    ))
])



## 11. Define helper functions for out-of-fold evaluation

This step produces:
- out-of-fold predicted probabilities for each model
- averaged test-set probabilities for submission
- cross-validated balanced accuracy and log loss

These out-of-fold predictions are also used later for weighted averaging and stacking.


In [12]:
def get_oof_predictions(model, X_data, y_data, X_test_data, cv):
    n_classes = len(np.unique(y_data))
    oof_pred = np.zeros((len(X_data), n_classes))
    test_pred = np.zeros((len(X_test_data), n_classes, cv.get_n_splits()))

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_data, y_data)):
        X_tr = X_data.iloc[train_idx].copy()
        X_val = X_data.iloc[val_idx].copy()
        y_tr = y_data[train_idx]

        model_fold = clone(model)
        model_fold.fit(X_tr, y_tr)

        oof_pred[val_idx] = model_fold.predict_proba(X_val)
        test_pred[:, :, fold] = model_fold.predict_proba(X_test_data)

        print(f"Finished fold {fold + 1}")

    return oof_pred, test_pred.mean(axis=2)

def summarize_oof(y_true, pred_proba, label):
    pred_class = pred_proba.argmax(axis=1)
    bal_acc = balanced_accuracy_score(y_true, pred_class)
    ll = log_loss(y_true, pred_proba, labels=np.arange(pred_proba.shape[1]))
    return pd.DataFrame({
        "approach": [label],
        "cv_balanced_accuracy": [bal_acc],
        "cv_log_loss": [ll]
    })



## 12. Evaluate the first round of individual models

This gives baseline model comparisons before targeted tuning.


In [13]:
oof_et, test_et = get_oof_predictions(model_et, X, y, X_test, skf)
oof_hgb, test_hgb = get_oof_predictions(model_hgb, X_fe, y, X_test_fe, skf)
oof_lr, test_lr = get_oof_predictions(model_lr, X_fe, y, X_test_fe, skf)

results_initial = pd.concat([
    summarize_oof(y, oof_et, "ExtraTrees on baseline features"),
    summarize_oof(y, oof_hgb, "HistGradientBoosting on engineered features"),
    summarize_oof(y, oof_lr, "LogisticRegression on engineered features"),
], ignore_index=True)

results_initial.sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])


Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5


,approach,cv_balanced_accuracy,cv_log_loss
1,HistGradientBoosting on engineered features,0.961912,0.060738
0,ExtraTrees on baseline features,0.813919,0.225513
2,LogisticRegression on engineered features,0.790878,0.274865



## 13. Tune the models beyond their default settings

The parameter search here is intentionally moderate so the notebook remains practical on a MacBook Air.
The goal is to show a real tuning effort and check whether changes improve performance.


In [14]:
et_candidates = [
    {"n_estimators": 180, "max_depth": None, "min_samples_leaf": 1},
    {"n_estimators": 220, "max_depth": None, "min_samples_leaf": 2},
    {"n_estimators": 260, "max_depth": 22, "min_samples_leaf": 1},
]

hgb_candidates = [
    {"learning_rate": 0.05, "max_depth": 6, "max_iter": 240, "min_samples_leaf": 25},
    {"learning_rate": 0.06, "max_depth": 8, "max_iter": 280, "min_samples_leaf": 25},
    {"learning_rate": 0.08, "max_depth": 6, "max_iter": 320, "min_samples_leaf": 35},
]

lr_candidates = [
    {"C": 0.5},
    {"C": 1.0},
    {"C": 2.0},
]

tuning_rows = []

for params in et_candidates:
    model = Pipeline(steps=[
        ("preprocessor", base_preprocessor),
        ("model", ExtraTreesClassifier(
            random_state=42,
            n_jobs=-1,
            **params
        ))
    ])
    oof_pred, _ = get_oof_predictions(model, X, y, X_test, skf)
    row = summarize_oof(y, oof_pred, f"ExtraTrees {params}").iloc[0].to_dict()
    row.update(params)
    tuning_rows.append(row)

for params in hgb_candidates:
    model = Pipeline(steps=[
        ("preprocessor", fe_preprocessor),
        ("model", HistGradientBoostingClassifier(
            random_state=42,
            l2_regularization=0.0,
            **params
        ))
    ])
    oof_pred, _ = get_oof_predictions(model, X_fe, y, X_test_fe, skf)
    row = summarize_oof(y, oof_pred, f"HistGradientBoosting {params}").iloc[0].to_dict()
    row.update(params)
    tuning_rows.append(row)

for params in lr_candidates:
    model = Pipeline(steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[
                ("num", Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]), numeric_cols_fe),
                ("cat", Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
                ]), categorical_cols_fe)
            ],
            remainder="drop"
        )),
        ("model", LogisticRegression(
            max_iter=2000,
            multi_class="auto",
            **params
        ))
    ])
    oof_pred, _ = get_oof_predictions(model, X_fe, y, X_test_fe, skf)
    row = summarize_oof(y, oof_pred, f"LogisticRegression {params}").iloc[0].to_dict()
    row.update(params)
    tuning_rows.append(row)

tuning_df = pd.DataFrame(tuning_rows)
tuning_df.sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True]).head(12)


Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5


,approach,cv_balanced_accuracy,cv_log_loss,n_estimators,max_depth,min_samples_leaf,learning_rate,max_iter,C
3,"HistGradientBoosting {'learning_rate': 0.05, '...",0.962054,0.059129,NaN,6.0,25.0,0.05,240.0,NaN
5,"HistGradientBoosting {'learning_rate': 0.08, '...",0.961946,0.061209,NaN,6.0,35.0,0.08,320.0,NaN
4,"HistGradientBoosting {'learning_rate': 0.06, '...",0.961912,0.060738,NaN,8.0,25.0,0.06,280.0,NaN
0,"ExtraTrees {'n_estimators': 180, 'max_depth': ...",0.814373,0.225858,180.0,NaN,1.0,NaN,NaN,NaN
1,"ExtraTrees {'n_estimators': 220, 'max_depth': ...",0.797880,0.231553,220.0,NaN,2.0,NaN,NaN,NaN
8,LogisticRegression {'C': 2.0},0.791174,0.274871,NaN,NaN,NaN,NaN,NaN,2.0
2,"ExtraTrees {'n_estimators': 260, 'max_depth': ...",0.791057,0.236668,260.0,22.0,1.0,NaN,NaN,NaN
7,LogisticRegression {'C': 1.0},0.790878,0.274865,NaN,NaN,NaN,NaN,NaN,1.0
6,LogisticRegression {'C': 0.5},0.790565,0.274868,NaN,NaN,NaN,NaN,NaN,0.5



## 14. Rebuild the best tuned versions of each model

This cell keeps the best parameter choice for each model family and saves the out-of-fold predictions for ensembling.


In [16]:
best_et_params = (
    tuning_df[tuning_df["approach"].str.contains("ExtraTrees")]
    .sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])
    .iloc[0][["n_estimators", "max_depth", "min_samples_leaf"]]
    .to_dict()
)

best_hgb_params = (
    tuning_df[tuning_df["approach"].str.contains("HistGradientBoosting")]
    .sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])
    .iloc[0][["learning_rate", "max_depth", "max_iter", "min_samples_leaf"]]
    .to_dict()
)

best_lr_params = (
    tuning_df[tuning_df["approach"].str.contains("LogisticRegression")]
    .sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])
    .iloc[0][["C"]]
    .to_dict()
)

best_et_model = Pipeline(steps=[
    ("preprocessor", base_preprocessor),
    ("model", ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1,
        n_estimators=int(best_et_params["n_estimators"]),
        max_depth=None if pd.isna(best_et_params["max_depth"]) else int(best_et_params["max_depth"]),
        min_samples_leaf=int(best_et_params["min_samples_leaf"])
    ))
])

best_hgb_model = Pipeline(steps=[
    ("preprocessor", fe_preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        l2_regularization=0.0,
        learning_rate=float(best_hgb_params["learning_rate"]),
        max_depth=int(best_hgb_params["max_depth"]),
        max_iter=int(best_hgb_params["max_iter"]),
        min_samples_leaf=int(best_hgb_params["min_samples_leaf"])
    ))
])

best_lr_model = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols_fe),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]), categorical_cols_fe)
        ],
        remainder="drop"
    )),
    ("model", LogisticRegression(
        C=float(best_lr_params["C"]),
        max_iter=2000,
        multi_class="auto"
    ))
])

best_oof_et, best_test_et = get_oof_predictions(best_et_model, X, y, X_test, skf)
best_oof_hgb, best_test_hgb = get_oof_predictions(best_hgb_model, X_fe, y, X_test_fe, skf)
best_oof_lr, best_test_lr = get_oof_predictions(best_lr_model, X_fe, y, X_test_fe, skf)

best_results = pd.concat([
    summarize_oof(y, best_oof_et, "Best ExtraTrees"),
    summarize_oof(y, best_oof_hgb, "Best HistGradientBoosting"),
    summarize_oof(y, best_oof_lr, "Best LogisticRegression"),
], ignore_index=True)

best_results.sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])


Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5
Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5


,approach,cv_balanced_accuracy,cv_log_loss
1,Best HistGradientBoosting,0.962054,0.059129
0,Best ExtraTrees,0.814373,0.225858
2,Best LogisticRegression,0.791174,0.274871



## 15. Evaluate which features are useful with feature importance

This first view uses built-in importance from the best ExtraTrees model on the original feature representation.


In [17]:
X_train_imp, X_valid_imp, y_train_imp, y_valid_imp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_et_model.fit(X_train_imp, y_train_imp)

preprocessor_fitted = best_et_model.named_steps["preprocessor"]
et_fitted = best_et_model.named_steps["model"]

feature_names = preprocessor_fitted.get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": et_fitted.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df.head(25)


,feature,importance
1,num__Soil_Moisture,0.200185
8,num__Wind_Speed_kmh,0.079443
4,num__Temperature_C,0.079417
22,cat__Crop_Growth_Stage_Harvest,0.072162
23,cat__Crop_Growth_Stage_Sowing,0.067630
21,cat__Crop_Growth_Stage_Flowering,0.063343
24,cat__Crop_Growth_Stage_Vegetative,0.060413
37,cat__Mulching_Used_Yes,0.037799
36,cat__Mulching_Used_No,0.037069
6,num__Rainfall_mm,0.032360



## 16. Evaluate which features are useful with permutation importance

Permutation importance is slower, so this is done on a validation split.
This helps check whether the most important features by the model are also important when they are perturbed.


In [20]:
perm = permutation_importance(
    best_et_model,
    X_valid_imp,
    y_valid_imp,
    n_repeats=3,
    random_state=42,
    scoring="balanced_accuracy",
    n_jobs=1
)

perm_df = pd.DataFrame({
    "feature": X_valid_imp.columns,
    "perm_importance_mean": perm.importances_mean,
    "perm_importance_std": perm.importances_std
}).sort_values("perm_importance_mean", ascending=False)

perm_df.head(25)


,feature,perm_importance_mean,perm_importance_std
11,Crop_Growth_Stage,0.253915,0.002313
2,Soil_Moisture,0.251956,0.001488
5,Temperature_C,0.154740,0.000559
16,Mulching_Used,0.152363,0.000808
9,Wind_Speed_kmh,0.131130,0.001219
7,Rainfall_mm,0.029407,0.000885
14,Water_Source,0.007384,0.001222
10,Crop_Type,0.006209,0.000763
17,Previous_Irrigation_mm,0.004641,0.000378
13,Irrigation_Type,0.004033,0.000879



## 17. Compare original and engineered feature representations

This helps answer whether the engineered features were useful in practice.


In [21]:
representation_comparison = pd.DataFrame({
    "representation": [
        "Original features with best ExtraTrees",
        "Engineered features with best HistGradientBoosting",
        "Engineered features with best LogisticRegression"
    ],
    "cv_balanced_accuracy": [
        summarize_oof(y, best_oof_et, "tmp")["cv_balanced_accuracy"].iloc[0],
        summarize_oof(y, best_oof_hgb, "tmp")["cv_balanced_accuracy"].iloc[0],
        summarize_oof(y, best_oof_lr, "tmp")["cv_balanced_accuracy"].iloc[0],
    ],
    "cv_log_loss": [
        summarize_oof(y, best_oof_et, "tmp")["cv_log_loss"].iloc[0],
        summarize_oof(y, best_oof_hgb, "tmp")["cv_log_loss"].iloc[0],
        summarize_oof(y, best_oof_lr, "tmp")["cv_log_loss"].iloc[0],
    ]
})

representation_comparison.sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])


,representation,cv_balanced_accuracy,cv_log_loss
1,Engineered features with best HistGradientBoos...,0.962054,0.059129
0,Original features with best ExtraTrees,0.814373,0.225858
2,Engineered features with best LogisticRegression,0.791174,0.274871



## 18. Combine models with an ensemble: equal-weight probability averaging

This is the simplest ensemble and often a strong reference point.


In [22]:
avg_oof_equal = (best_oof_et + best_oof_hgb + best_oof_lr) / 3
avg_test_equal = (best_test_et + best_test_hgb + best_test_lr) / 3

avg_equal_results = summarize_oof(y, avg_oof_equal, "Equal-weight average ensemble")
avg_equal_results


,approach,cv_balanced_accuracy,cv_log_loss
0,Equal-weight average ensemble,0.918426,0.1571



## 19. Combine models with an ensemble: custom weighted averaging

The weights were adjusted based on which models performed best individually.


In [24]:
weighted_oof = (
    0.85 * best_oof_hgb +
    0.10 * best_oof_et +
    0.05 * best_oof_lr
)

weighted_test = (
    0.85 * best_test_hgb +
    0.10 * best_test_et +
    0.05 * best_test_lr
)

weighted_results = summarize_oof(y, weighted_oof, "Custom weighted average ensemble")
weighted_results


,approach,cv_balanced_accuracy,cv_log_loss
0,Custom weighted average ensemble,0.961599,0.07424



## 20. Combine models with an ensemble: stacking

This uses out-of-fold class probabilities from the base models as inputs to a multinomial logistic regression meta-model.


In [25]:
meta_X = np.hstack([best_oof_et, best_oof_hgb, best_oof_lr])
meta_X_test = np.hstack([best_test_et, best_test_hgb, best_test_lr])

meta_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    multi_class="auto"
)

meta_model.fit(meta_X, y)
stack_oof = meta_model.predict_proba(meta_X)
stack_test = meta_model.predict_proba(meta_X_test)

stack_results = summarize_oof(y, stack_oof, "Stacking with multinomial logistic regression")
stack_results


,approach,cv_balanced_accuracy,cv_log_loss
0,Stacking with multinomial logistic regression,0.960908,0.067706



## 21. Compare the base models and ensembles

This table is the main comparison point for the notebook.


In [26]:
all_results = pd.concat([
    best_results,
    avg_equal_results,
    weighted_results,
    stack_results
], ignore_index=True)

all_results.sort_values(["cv_balanced_accuracy", "cv_log_loss"], ascending=[False, True])


,approach,cv_balanced_accuracy,cv_log_loss
1,Best HistGradientBoosting,0.962054,0.059129
4,Custom weighted average ensemble,0.961599,0.074240
5,Stacking with multinomial logistic regression,0.960908,0.067706
3,Equal-weight average ensemble,0.918426,0.157100
0,Best ExtraTrees,0.814373,0.225858
2,Best LogisticRegression,0.791174,0.274871



## 22. Create submission files for the main approaches

This notebook writes out several submissions so different approaches can be compared on the Kaggle leaderboard.


In [27]:
def probs_to_submission(test_proba, filename="submission.csv"):
    pred_labels = le.inverse_transform(np.argmax(test_proba, axis=1))
    submission = sample_submission.copy()
    submission[target_col] = pred_labels
    submission.to_csv(filename, index=False)
    return submission

submission_best_et = probs_to_submission(best_test_et, "submission_best_et.csv")
submission_best_hgb = probs_to_submission(best_test_hgb, "submission_best_hgb.csv")
submission_best_lr = probs_to_submission(best_test_lr, "submission_best_lr.csv")
submission_avg_equal = probs_to_submission(avg_test_equal, "submission_avg_equal.csv")
submission_avg_weighted = probs_to_submission(weighted_test, "submission_avg_weighted.csv")
submission_stack = probs_to_submission(stack_test, "submission_stacked.csv")

print("Saved files:")
print("submission_best_et.csv")
print("submission_best_hgb.csv")
print("submission_best_lr.csv")
print("submission_avg_equal.csv")
print("submission_avg_weighted.csv")
print("submission_stacked.csv")

submission_stack.head()


Saved files:
submission_best_et.csv
submission_best_hgb.csv
submission_best_lr.csv
submission_avg_equal.csv
submission_avg_weighted.csv
submission_stacked.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low



## 23. Leaderboard tracking table

This table is included so public leaderboard scores can be added after each submission.


In [28]:
all_results = pd.DataFrame({
    "approach": [
        "Best HistGradientBoosting",
        "Custom weighted average ensemble",
        "Stacking with multinomial logistic regression",
        "Equal-weight average ensemble",
        "Best ExtraTrees",
        "Best LogisticRegression"
    ],
    "cv_balanced_accuracy": [
        0.962054,
        0.961599,
        0.960908,
        0.918426,
        0.814373,
        0.791174
    ],
    "cv_log_loss": [
        0.059129,
        0.074240,
        0.067706,
        0.157100,
        0.225858,
        0.274871
    ],
    "public_leaderboard_score": [
        0.95993,   # best_hgb
        0.95940,   # weighted
        0.95896,   # stacked
        0.91678,   # equal
        0.81364,   # et
        0.79038    # lr
    ]
})

all_results.sort_values(
    ["public_leaderboard_score", "cv_balanced_accuracy"],
    ascending=[False, False]
)

,approach,cv_balanced_accuracy,cv_log_loss,public_leaderboard_score
0,Best HistGradientBoosting,0.962054,0.059129,0.95993
1,Custom weighted average ensemble,0.961599,0.074240,0.95940
2,Stacking with multinomial logistic regression,0.960908,0.067706,0.95896
3,Equal-weight average ensemble,0.918426,0.157100,0.91678
4,Best ExtraTrees,0.814373,0.225858,0.81364
5,Best LogisticRegression,0.791174,0.274871,0.79038



## 24. Discussion

### What I tried

built three different models that differ in both model type and feature representation. ExtraTrees was trained on the original feature set and served as a baseline tree-based model. HistGradientBoosting was trained on the engineered feature set and was the strongest model. LogisticRegression was trained on the engineered feature set with scaling and provided a simpler linear comparison.

For feature engineering, I created multiple types of features. These included interaction terms such as soil moisture with temperature and humidity with temperature. I created rainfall and irrigation combinations, including both sums and differences. I added a water input per area feature using rainfall, irrigation, and field area. I also created ratio features such as sunlight divided by temperature. Additional features included log transforms for selected variables, row-level summary statistics, categorical combinations such as soil type with crop type and irrigation type with water source, and frequency encoding for categorical variables.

This approach allowed me to test different representations of the data rather than relying on a single pattern of feature engineering.

### What worked well

The HistGradientBoosting model performed the best across both cross-validation and the public leaderboard. It achieved a cross-validated balanced accuracy of 0.962054 and a leaderboard score of 0.95993. This indicates that the engineered feature set combined with gradient boosting captured the most predictive signal.

The custom weighted average ensemble also performed well. It achieved a cross-validated balanced accuracy of 0.961599 and a leaderboard score of 0.95940. This was a large improvement over equal-weight averaging, which only achieved 0.918426 in cross-validation and 0.91678 on the leaderboard.

The improvement from equal-weight averaging to weighted averaging shows that assigning higher weight to the strongest model was effective.

### What did not work well

Stacking did not outperform the best individual model or the weighted ensemble. The stacking model achieved a cross-validated balanced accuracy of 0.960908 and a leaderboard score of 0.95896. This is slightly lower than both the HistGradientBoosting model and the weighted ensemble.

Equal-weight averaging performed significantly worse than the other ensemble methods. This is because weaker models, specifically ExtraTrees and LogisticRegression, reduced the overall performance when given equal influence.

The ExtraTrees and LogisticRegression models individually performed much worse than the HistGradientBoosting model. ExtraTrees achieved 0.814373 cross-validation and 0.81364 on the leaderboard. LogisticRegression achieved 0.791174 cross-validation and 0.79038 on the leaderboard. These models did not contribute enough additional information to improve ensemble performance.

### Feature importance insights

Permutation importance showed that the most important features included Crop_Growth_Stage, Soil_Moisture, Temperature_C, Mulching_Used, and Wind_Speed_kmh. These features are directly related to irrigation conditions and align with expectations for this problem.

Moderately important features included Rainfall_mm, Water_Source, Crop_Type, and Previous_Irrigation_mm. These also reflect water availability and crop characteristics.

Lower importance features included Field_Area_hectare and Season. These had very small importance values and likely contributed less to prediction performance.

Overall, the feature importance results indicate that both environmental conditions and crop growth stage are key drivers of irrigation need.

### What I will continue to use

I will continue to use a structured workflow that includes building a baseline model, creating multiple types of engineered features, and comparing feature representations directly. I will continue to use cross-validation for model evaluation and permutation importance for understanding feature contributions.

I will also continue to use weighted averaging as a primary ensemble method. It is simple to implement and performed well in this case.

### Were the improvements meaningful or small

The improvement from equal-weight averaging to weighted averaging was meaningful. Balanced accuracy increased from 0.918426 to 0.961599, and the leaderboard score increased from 0.91678 to 0.95940.

However, the improvement from the best individual model to the ensemble was small. The HistGradientBoosting model achieved 0.962054 in cross-validation and 0.95993 on the leaderboard, which is slightly higher than both the weighted ensemble and the stacking model.

This indicates that most of the predictive signal was captured by the HistGradientBoosting model, and the additional models did not provide enough new information to significantly improve performance.

### Final summary

The HistGradientBoosting model with engineered features was the best overall approach. Weighted averaging provided a strong ensemble improvement over equal weighting but did not surpass the best model. Stacking did not provide additional benefit. The results were consistent between cross-validation and the public leaderboard, which indicates that the modeling approach and validation strategy were reliable.
